In [124]:
import pandas as pd
from math import prod
import random
random.seed(17)
pd.set_option('display.max_rows',20)
import time

In [125]:
num_activities = 4
max_tries = 3
prob_blink = 0.1

In [126]:
start_time = time.time()
print(start_time)

1778232578.3333786


In [127]:
pdf_occ = pd.DataFrame({'occlusion':['easy','med','hard'],
                        'prob' : [0.5,0.3,0.2]})
pdf_occ

,occlusion,prob
0,easy,0.5
1,med,0.3
2,hard,0.2


In [128]:
sum(pdf_occ['prob'])==1.0

True

In [129]:
def validate_pdf_size(pdf):
    assert len(pdf)==prod([(len(pdf.drop('prob',axis=1).drop_duplicates(col))) for col in pdf.drop('prob',axis=1).columns])

In [130]:
def validate_pdf(pdf, tol=1e-10, all_combinations=False):
    if all_combinations:
        validate_pdf_size(pdf)
    total_prob = sum(pdf['prob'])
    assert abs(total_prob-1.0)<tol, f'Probability sums to {total_prob}'
validate_pdf(pdf_occ)

In [131]:
pdf_app = pd.DataFrame({'appearance':['blend','med','obvious'],
                        'prob' : [0.2,0.4,0.4]})
validate_pdf(pdf_app)
pdf_app

,appearance,prob
0,blend,0.2
1,med,0.4
2,obvious,0.4


In [132]:
def joint_pdf_independent(pdf1,pdf2,prune=True):
    jpdf = pd.DataFrame({},columns=pdf1.columns.union(pdf2.columns))
    for i,r1 in pdf1.iterrows():
        for j,r2 in pdf2.iterrows():
            new_row = pd.concat((r1.drop('prob'),r2.drop('prob'),pd.Series({'prob':r1['prob']*r2['prob']}))).to_frame().T
            jpdf = pd.concat((jpdf,new_row))
    if prune==True:
        jpdf = jpdf[jpdf['prob']>0.0]
    return jpdf

In [133]:
def joint_pdf_independent(pdf1,pdf2,prune=True):
    jpdf = pd.merge(pdf1,pdf2,how='cross')
    jpdf['prob'] = jpdf['prob_x']*jpdf['prob_y']
    if prune==True:
        jpdf = jpdf[jpdf['prob']>0.0]
    return jpdf.drop(['prob_x','prob_y'],axis=1)

In [134]:
jpdf_app_occ = joint_pdf_independent(pdf_app,pdf_occ)
validate_pdf(jpdf_app_occ)
jpdf_app_occ

,appearance,occlusion,prob
0,blend,easy,0.10
1,blend,med,0.06
2,blend,hard,0.04
3,med,easy,0.20
4,med,med,0.12
5,med,hard,0.08
6,obvious,easy,0.20
7,obvious,med,0.12
8,obvious,hard,0.08


In [135]:
def validate_cpdf(cpdf, all_combinations=True):
    if all_combinations:
        validate_pdf_size(cpdf)
    givens = [c for c in cpdf.columns if c.startswith('|')]
    for i,r in cpdf[givens].drop_duplicates().iterrows():
        assert sum(cpdf[cpdf.eq(r)[givens].all(axis=1)]['prob'])==1

In [136]:
cpdf_indis_a_given_app = pd.DataFrame({'indis_a':[True,True,True,False,False,False],
                                       '|appearance':['blend','med','obvious','blend','med','obvious'],
                                       'prob' : [0.9,0.5,0.1,0.1,0.5,0.9]})
validate_cpdf(cpdf_indis_a_given_app)
cpdf_indis_a_given_app

,indis_a,|appearance,prob
0,True,blend,0.9
1,True,med,0.5
2,True,obvious,0.1
3,False,blend,0.1
4,False,med,0.5
5,False,obvious,0.9


In [137]:
# never indistinguishable
#cpdf_indis_a_given_app = pd.DataFrame({'indis_a':[True,True,True,False,False,False],
#                                       '|appearance':['blend','med','obvious','blend','med','obvious'],
#                                       'prob' : [0,0,0,1,1,1]})
#validate_cpdf(cpdf_indis_a_given_app)
#cpdf_indis_a_given_app

In [138]:
def joint_pdf_dependent(pdf,cpdf,prune=True):
    validate_cpdf(cpdf)
    validate_pdf(pdf)
    givens = [c for c in cpdf.columns if c.startswith('|')]
    givens_nobar = [col.strip('|') for col in givens]
    assert all([col in pdf.columns for col in givens_nobar])
    new_vars = [c for c in cpdf.drop('prob',axis=1).columns if not c.startswith('|')]
    jpdf1 = pd.merge(cpdf[new_vars].drop_duplicates(),pdf,how='cross')
    jpdf2 = pd.merge(jpdf1,cpdf.rename(columns=dict(zip(givens,givens_nobar))),on=givens_nobar+new_vars)
    jpdf2['prob'] = jpdf2['prob_x']*jpdf2['prob_y']
    if prune==True:
        jpdf2 = jpdf2[jpdf2['prob']>0.0]
    return jpdf2.drop(['prob_x','prob_y'],axis=1)

In [139]:
jpdf_indis_app_occ = joint_pdf_dependent(jpdf_app_occ,cpdf_indis_a_given_app)
validate_pdf(jpdf_indis_app_occ)
jpdf_indis_app_occ

,indis_a,appearance,occlusion,prob
0,True,blend,easy,0.090
1,True,blend,med,0.054
2,True,blend,hard,0.036
3,True,med,easy,0.100
4,True,med,med,0.060
5,True,med,hard,0.040
6,True,obvious,easy,0.020
7,True,obvious,med,0.012
8,True,obvious,hard,0.008
9,False,blend,easy,0.010


In [140]:
def get_conditional_pdf(jpdf,of,given):
    validate_pdf(jpdf)
    assert all([c in jpdf.columns for c in of])
    assert all([c in jpdf.columns for c in given])
    cpdf = pd.DataFrame({},columns=of+['|'+s for s in given])
    for i,r1 in jpdf[given].drop_duplicates().iterrows():
        given_rows = jpdf[jpdf.eq(r1)[given].all(axis=1)]
        given_prob = sum(given_rows['prob'])
        for j,r2 in jpdf[of].drop_duplicates().iterrows():
            of_rows = given_rows[given_rows.eq(r2)[of].all(axis=1)]
            of_prob = sum(of_rows['prob'])
            cond_prob = of_prob/given_prob
            new_row = pd.concat((r1.rename(lambda s : '|'+s),r2,pd.Series({'prob':cond_prob}))).to_frame().T
            cpdf = pd.concat((cpdf,new_row))
    return cpdf 


In [141]:
def get_marginal_pdf(jpdf,of):
    validate_pdf(jpdf)
    assert all([c in jpdf.columns for c in of])
    mpdf = pd.DataFrame({},columns=of)
    for i,r1 in jpdf[of].drop_duplicates().iterrows():
        of_rows = jpdf[jpdf.eq(r1)[of].all(axis=1)]
        of_prob = sum(of_rows['prob'])
        new_row = pd.concat((r1,pd.Series({'prob':of_prob}))).to_frame().T
        mpdf = pd.concat((mpdf,new_row))
    return mpdf 

In [142]:
def get_marginal_pdf(jpdf,of):
    validate_pdf(jpdf)
    assert all([c in jpdf.columns for c in of])
    mpdf = jpdf[of].drop_duplicates()[of]
    def marg_prob(r):
        of_rows = jpdf[jpdf[of].eq(r).all(axis=1)]
        return sum(of_rows['prob'])
    mpdf['prob'] = mpdf.apply(marg_prob,axis=1)
    return mpdf 

In [143]:
def get_marginal_pdf(jpdf,of):
    validate_pdf(jpdf)
    assert all([c in jpdf.columns for c in of])
    margin_cols = [c for c in jpdf.columns if (c!='prob') and (c not in of)]
    #print(margin_cols)
    pivot_jpdf = jpdf.pivot(index=of,columns=margin_cols,values='prob')
    return pivot_jpdf.sum(axis=1).reset_index().rename(columns={0:'prob'})

In [144]:
get_marginal_pdf(jpdf_indis_app_occ,of=['indis_a','appearance'])

,indis_a,appearance,prob
0,False,blend,0.02
1,False,med,0.20
2,False,obvious,0.36
3,True,blend,0.18
4,True,med,0.20
5,True,obvious,0.04


In [145]:
def get_conditional_pdf(jpdf,of,given):
    validate_pdf(jpdf)
    assert all([c in jpdf.columns for c in of])
    assert all([c in jpdf.columns for c in given])
    jmpdf1 = get_marginal_pdf(jpdf,of + given)
    jmpdf2 = get_marginal_pdf(jmpdf1,given)
    cpdf1 = pd.merge(jmpdf1,jmpdf2,on=given)
    cpdf1['prob'] = cpdf1['prob_x']/cpdf1['prob_y']
    renaming = dict([(col,'|'+col) for col in given])
    return cpdf1.drop(['prob_x','prob_y'],axis=1).rename(columns=renaming)


In [146]:
cpdf_occ_a_given_occ = {'circle':pd.DataFrame({'occ_a':[True,True,True,False,False,False],
                                               '|occlusion':['hard','med','easy','hard','med','easy'],
                                               'prob' : [0.5,0.3,0.1,0.5,0.7,0.9]}),
                        'mower':pd.DataFrame({'occ_a':[True,True,True,False,False,False],
                                               '|occlusion':['hard','med','easy','hard','med','easy'],
                                               'prob' : [0.7,0.5,0.3,0.3,0.5,0.7]}),
                        'along':pd.DataFrame({'occ_a':[True,True,True,False,False,False],
                                               '|occlusion':['hard','med','easy','hard','med','easy'],
                                               'prob' : [0.9,0.7,0.5,0.1,0.3,0.5]}),
}
for k in cpdf_occ_a_given_occ:
    validate_cpdf(cpdf_occ_a_given_occ[k])
cpdf_occ_a_given_occ

{'circle':    occ_a |occlusion  prob
 0   True       hard   0.5
 1   True        med   0.3
 2   True       easy   0.1
 3  False       hard   0.5
 4  False        med   0.7
 5  False       easy   0.9,
 'mower':    occ_a |occlusion  prob
 0   True       hard   0.7
 1   True        med   0.5
 2   True       easy   0.3
 3  False       hard   0.3
 4  False        med   0.5
 5  False       easy   0.7,
 'along':    occ_a |occlusion  prob
 0   True       hard   0.9
 1   True        med   0.7
 2   True       easy   0.5
 3  False       hard   0.1
 4  False        med   0.3
 5  False       easy   0.5}

In [147]:
# never occluded
#cpdf_occ_a_given_occ = {'circle':pd.DataFrame({'occ_a':[True,True,True,False,False,False],
#                                               '|occlusion':['hard','med','easy','hard','med','easy'],
#                                               'prob' : [0.,0.,0.,1.0,1.0,1.0]}),
#                        'mower':pd.DataFrame({'occ_a':[True,True,True,False,False,False],
#                                               '|occlusion':['hard','med','easy','hard','med','easy'],
#                                               'prob' : [0.,0.,0.,1.0,1.0,1.0]}),
#                        'along':pd.DataFrame({'occ_a':[True,True,True,False,False,False],
#                                               '|occlusion':['hard','med','easy','hard','med','easy'],
#                                               'prob' : [0.,0.,0.,1.0,1.0,1.0]}),
#}
#for k in cpdf_occ_a_given_occ:
#    validate_cpdf(cpdf_occ_a_given_occ[k])
#cpdf_occ_a_given_occ

In [148]:
#num_activities = 4
activities = [f'a{i+1}' for i in range(num_activities)]
print(activities)

['a1', 'a2', 'a3', 'a4']


In [149]:
activity_types = {}
for a in activities:
    activity_types[a] = random.sample(['along','mower','circle'],1)[0]
#activity_types = {'a1': 'circle', 'a2': 'mower', 'a3': 'mower'} # overwrite
print(activity_types)

{'a1': 'circle', 'a2': 'mower', 'a3': 'mower', 'a4': 'mower'}


In [150]:
jpdf_app_occ

,appearance,occlusion,prob
0,blend,easy,0.10
1,blend,med,0.06
2,blend,hard,0.04
3,med,easy,0.20
4,med,med,0.12
5,med,hard,0.08
6,obvious,easy,0.20
7,obvious,med,0.12
8,obvious,hard,0.08


In [151]:
jpdf_model = jpdf_app_occ
for a in activities:
    this_occ_cpdf = cpdf_occ_a_given_occ[activity_types[a]].rename(columns={'occ_a':'occ_'+a})
    jpdf_model = joint_pdf_dependent(jpdf_model,this_occ_cpdf)
    this_ind_cpdf = cpdf_indis_a_given_app.rename(columns={'indis_a':'indis_'+a})
    jpdf_model = joint_pdf_dependent(jpdf_model,this_ind_cpdf)
validate_pdf(jpdf_model)
jpdf_model

,indis_a4,occ_a4,indis_a3,occ_a3,indis_a2,occ_a2,indis_a1,occ_a1,appearance,occlusion,prob
0,True,True,True,True,True,True,True,True,blend,easy,0.000177
1,True,True,True,True,True,True,True,True,blend,med,0.001476
2,True,True,True,True,True,True,True,True,blend,hard,0.004501
3,True,True,True,True,True,True,True,True,med,easy,0.000034
4,True,True,True,True,True,True,True,True,med,med,0.000281
...,...,...,...,...,...,...,...,...,...,...,...
2299,False,False,False,False,False,False,False,False,med,med,0.000656
2300,False,False,False,False,False,False,False,False,med,hard,0.000068
2301,False,False,False,False,False,False,False,False,obvious,easy,0.040508
2302,False,False,False,False,False,False,False,False,obvious,med,0.006889


In [152]:
num_regions = 2**num_activities
coverage = {}
for i in range(num_regions):
    if i>0:
        coverage[f'r{i}'] = [a for j,a in enumerate(activities) if 2**j & i > 0]
regions = [r for r in coverage.keys()]
print(regions)
print(coverage)

['r1', 'r2', 'r3', 'r4', 'r5', 'r6', 'r7', 'r8', 'r9', 'r10', 'r11', 'r12', 'r13', 'r14', 'r15']
{'r1': ['a1'], 'r2': ['a2'], 'r3': ['a1', 'a2'], 'r4': ['a3'], 'r5': ['a1', 'a3'], 'r6': ['a2', 'a3'], 'r7': ['a1', 'a2', 'a3'], 'r8': ['a4'], 'r9': ['a1', 'a4'], 'r10': ['a2', 'a4'], 'r11': ['a1', 'a2', 'a4'], 'r12': ['a3', 'a4'], 'r13': ['a1', 'a3', 'a4'], 'r14': ['a2', 'a3', 'a4'], 'r15': ['a1', 'a2', 'a3', 'a4']}


In [153]:
region_probs = [0.75+random.random() for r in coverage]
region_probs = [p / sum(region_probs) for p in region_probs]
#region_probs = [0,0,0,0,0,0,1] # always vis to all
assert(sum(region_probs)==1)
region_probs

[0.05414156592747125,
 0.07895577825656366,
 0.0757327999851681,
 0.0735019464472059,
 0.044795482368756544,
 0.04046128031312302,
 0.0590653142401742,
 0.07792920498055463,
 0.052202098942740034,
 0.0651809103095121,
 0.05558983895944513,
 0.083133491321297,
 0.08828223612564413,
 0.05997250352458498,
 0.09105554829775936]

In [154]:
pdf_region = pd.DataFrame({'region': coverage.keys(),
                           'prob': region_probs})
validate_pdf(pdf_region)
pdf_region

,region,prob
0,r1,0.054142
1,r2,0.078956
2,r3,0.075733
3,r4,0.073502
4,r5,0.044795
5,r6,0.040461
6,r7,0.059065
7,r8,0.077929
8,r9,0.052202
9,r10,0.065181


In [155]:
jpdf_model2 = joint_pdf_independent(jpdf_model,pdf_region)
validate_pdf(jpdf_model2)
jpdf_model2

,indis_a4,occ_a4,indis_a3,occ_a3,indis_a2,occ_a2,indis_a1,occ_a1,appearance,occlusion,region,prob
0,True,True,True,True,True,True,True,True,blend,easy,r1,0.000010
1,True,True,True,True,True,True,True,True,blend,easy,r2,0.000014
2,True,True,True,True,True,True,True,True,blend,easy,r3,0.000013
3,True,True,True,True,True,True,True,True,blend,easy,r4,0.000013
4,True,True,True,True,True,True,True,True,blend,easy,r5,0.000008
...,...,...,...,...,...,...,...,...,...,...,...,...
34555,False,False,False,False,False,False,False,False,obvious,hard,r11,0.000039
34556,False,False,False,False,False,False,False,False,obvious,hard,r12,0.000059
34557,False,False,False,False,False,False,False,False,obvious,hard,r13,0.000063
34558,False,False,False,False,False,False,False,False,obvious,hard,r14,0.000042


In [156]:
jpdf_model3 = jpdf_model2
cpdf_vis_a = {}
for a in activities:
    print('Activity ',a)
    cpdf_vis_a[a] = pd.DataFrame({},columns=['vis_'+a,'|region','|occ_'+a,'|indis_'+a])
    for r in regions:
        for occ_a in [True,False]:
            for indis_a in [True,False]:
                prob_true = 0.0
                if occ_a==False:
                    if indis_a==False:
                        if a in coverage[r]:
                            prob_true = 1.0
                new_rows = pd.DataFrame({'vis_'+a:[True,False],
                                         'prob':[prob_true,1-prob_true]})
                new_rows['|occ_'+a] = occ_a
                new_rows['|indis_'+a] = indis_a
                new_rows['|region'] = r
                cpdf_vis_a[a] = pd.concat((cpdf_vis_a[a],new_rows))
    validate_cpdf(cpdf_vis_a[a])
    print('Conditional distribution is ',cpdf_vis_a[a].shape)
    jpdf_model3 = joint_pdf_dependent(jpdf_model3,cpdf_vis_a[a])
    print('Model is ', jpdf_model3.shape)          
validate_pdf(jpdf_model3)
jpdf_model3

Activity  a1
Conditional distribution is  (120, 5)
Model is  (34560, 13)
Activity  a2
Conditional distribution is  (120, 5)
Model is  (34560, 14)
Activity  a3
Conditional distribution is  (120, 5)
Model is  (34560, 15)
Activity  a4
Conditional distribution is  (120, 5)
Model is  (34560, 16)


,vis_a4,vis_a3,vis_a2,vis_a1,indis_a4,occ_a4,indis_a3,occ_a3,indis_a2,occ_a2,indis_a1,occ_a1,appearance,occlusion,region,prob
55,True,True,True,True,False,False,False,False,False,False,False,False,blend,easy,r15,2.810885e-07
57,True,True,True,True,False,False,False,False,False,False,False,False,blend,med,r15,4.780416e-08
59,True,True,True,True,False,False,False,False,False,False,False,False,blend,hard,r15,4.917000e-09
61,True,True,True,True,False,False,False,False,False,False,False,False,med,easy,r15,3.513606e-04
63,True,True,True,True,False,False,False,False,False,False,False,False,med,med,r15,5.975520e-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69096,False,False,False,False,False,False,False,False,False,False,False,True,med,med,r1,1.522732e-05
69099,False,False,False,False,False,False,False,False,False,False,False,True,med,hard,r1,3.654556e-06
69102,False,False,False,False,False,False,False,False,False,False,False,True,obvious,easy,r1,2.436829e-04
69105,False,False,False,False,False,False,False,False,False,False,False,True,obvious,med,r1,1.598503e-04


In [157]:
jpdf_model3a = get_marginal_pdf(jpdf_model3,['appearance','region','occlusion'] + ['vis_'+a for a in activities])
validate_pdf(jpdf_model3a)
jpdf_model3a

,appearance,region,occlusion,vis_a1,vis_a2,vis_a3,vis_a4,prob
0,blend,r1,easy,False,False,False,False,0.004927
1,blend,r1,easy,True,False,False,False,0.000487
2,blend,r1,hard,False,False,False,False,0.002057
3,blend,r1,hard,True,False,False,False,0.000108
4,blend,r1,med,False,False,False,False,0.003021
...,...,...,...,...,...,...,...,...
715,obvious,r9,hard,True,False,False,True,0.000507
716,obvious,r9,med,False,False,False,False,0.001275
717,obvious,r9,med,False,False,False,True,0.001043
718,obvious,r9,med,True,False,False,False,0.002171


In [158]:
#max_tries = 2
num_tries = [i for i in range(max_tries+1)]
prob_tries = [1/len(num_tries) for i in num_tries]
pdf_tries = pd.DataFrame({'num_a':num_tries,
                          'prob': prob_tries})
validate_pdf(pdf_tries)
pdf_tries

,num_a,prob
0,0,0.25
1,1,0.25
2,2,0.25
3,3,0.25


In [159]:
cpdf_det_a_given_vis_a_num_a = pd.merge(pd.DataFrame({'det_a':[True,False]}),
                                        pd.DataFrame({'|vis_a':[True,False]}),how='cross')
cpdf_det_a_given_vis_a_num_a = pd.merge(cpdf_det_a_given_vis_a_num_a,
                                        pd.DataFrame({'|num_a':[i for i in range(max_tries+1)]}),how='cross')
cpdf_det_a_given_vis_a_num_a['prob'] = 0.0
def cond_prob_det(r,prob_blink=prob_blink):
    prob_true = 0.0
    if r['|vis_a']==True:
        if r['|num_a']>0:
            prob_true = (1-prob_blink**r['|num_a'])
    if r['det_a']==True:
        return prob_true
    else:
        return 1-prob_true
cpdf_det_a_given_vis_a_num_a['prob'] = cpdf_det_a_given_vis_a_num_a.apply(cond_prob_det,axis=1)
validate_cpdf(cpdf_det_a_given_vis_a_num_a)
cpdf_det_a_given_vis_a_num_a

,det_a,|vis_a,|num_a,prob
0,True,True,0,0.000
1,True,True,1,0.900
2,True,True,2,0.990
3,True,True,3,0.999
4,True,False,0,0.000
5,True,False,1,0.000
6,True,False,2,0.000
7,True,False,3,0.000
8,False,True,0,1.000
9,False,True,1,0.100


In [160]:
jpdf_model4 = jpdf_model3a
for a in activities:
    this_tries_pdf = pdf_tries.rename(columns={'num_a':'num_'+a})
    jpdf_model4 = joint_pdf_independent(jpdf_model4,this_tries_pdf)
    this_det_cpdf = cpdf_det_a_given_vis_a_num_a.rename(columns={'|num_a':'|num_'+a,
                                                                '|vis_a':'|vis_'+a,
                                                                'det_a':'det_'+a})
    jpdf_model4 = joint_pdf_dependent(jpdf_model4,this_det_cpdf)
validate_pdf(jpdf_model4)
jpdf_model4

,det_a4,det_a3,det_a2,det_a1,appearance,region,occlusion,vis_a1,vis_a2,vis_a3,vis_a4,num_a1,num_a2,num_a3,num_a4,prob
109,True,True,True,True,blend,r15,easy,True,True,True,True,1,1,1,1,7.203990e-10
110,True,True,True,True,blend,r15,easy,True,True,True,True,1,1,1,2,7.924389e-10
111,True,True,True,True,blend,r15,easy,True,True,True,True,1,1,1,3,7.996429e-10
113,True,True,True,True,blend,r15,easy,True,True,True,True,1,1,2,1,7.924389e-10
114,True,True,True,True,blend,r15,easy,True,True,True,True,1,1,2,2,8.716828e-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
724387,False,False,False,False,obvious,r9,med,True,False,False,True,3,3,2,3,6.937170e-12
724388,False,False,False,False,obvious,r9,med,True,False,False,True,3,3,3,0,6.937170e-09
724389,False,False,False,False,obvious,r9,med,True,False,False,True,3,3,3,1,6.937170e-10
724390,False,False,False,False,obvious,r9,med,True,False,False,True,3,3,3,2,6.937170e-11


In [161]:
detect_cols = ['det_'+a for a in activities]
jpdf_model4['det_any']=jpdf_model4[detect_cols].sum(axis=1)>0
jpdf_model4

,det_a4,det_a3,det_a2,det_a1,appearance,region,occlusion,vis_a1,vis_a2,vis_a3,vis_a4,num_a1,num_a2,num_a3,num_a4,prob,det_any
109,True,True,True,True,blend,r15,easy,True,True,True,True,1,1,1,1,7.203990e-10,True
110,True,True,True,True,blend,r15,easy,True,True,True,True,1,1,1,2,7.924389e-10,True
111,True,True,True,True,blend,r15,easy,True,True,True,True,1,1,1,3,7.996429e-10,True
113,True,True,True,True,blend,r15,easy,True,True,True,True,1,1,2,1,7.924389e-10,True
114,True,True,True,True,blend,r15,easy,True,True,True,True,1,1,2,2,8.716828e-10,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
724387,False,False,False,False,obvious,r9,med,True,False,False,True,3,3,2,3,6.937170e-12,False
724388,False,False,False,False,obvious,r9,med,True,False,False,True,3,3,3,0,6.937170e-09,False
724389,False,False,False,False,obvious,r9,med,True,False,False,True,3,3,3,1,6.937170e-10,False
724390,False,False,False,False,obvious,r9,med,True,False,False,True,3,3,3,2,6.937170e-11,False


In [162]:
summary_table = get_conditional_pdf(jpdf_model4,of=['det_any'],given=['num_'+a for a in activities])

In [163]:
summary_table[summary_table['det_any']==True].drop('det_any',axis=1).sort_values(['prob'],ascending=[False])

,|num_a1,|num_a2,|num_a3,|num_a4,prob
510,3,3,3,3,0.529039
494,3,2,3,3,0.528350
506,3,3,2,3,0.528348
509,3,3,3,2,0.528289
446,2,3,3,3,0.528072
...,...,...,...,...,...
303,0,3,0,0,0.170678
287,0,2,0,0,0.169141
256,0,0,0,1,0.167600
259,0,0,1,0,0.157931


In [164]:
end_time = time.time()
print(f'Ran in {end_time - start_time:.1f} seconds')

Ran in 2.0 seconds
